---

#**Project Name: Utilizing Image Processing Model into ERP Systems for Product Searching in Fashion E-Commerce Platform**
# Group 7 - Research
Programming Language: Pytorch

Applied Model: ResNet50

Model Developers: Ngọc Quỳnh, Anh Hào

Data Annotators: Nhật Huyền, Uy Minh

---


**Cài đặt thư viện và Kết nối Drive**

In [1]:
# Cài đặt thư viện tìm kiếm vector FAISS
!pip install faiss-cpu

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import numpy as np
import faiss
import os
from PIL import Image

# Kết nối Drive để lấy dataset.zip
from google.colab import drive
drive.mount('/content/drive')

# Giải nén dataset vào Colab (thay đúng tên file zip của bạn)
!unzip /content/drive/MyDrive/dataset.zip -d /content/

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 30.7 MB/s eta 0:00:00
Mounted at /content/drive
Archive:  /content/drive/MyDrive/dataset.zip
   creating: /content/dataset_fashion/
  inflating: /content/dataset_fashion/dataset.csv  
   creating: /content/dataset_fashion/Dress/
  inflating: /content/dataset_fashion/Dress/SP161.jpg  
  inflating: /content/dataset_fashion/Dress/SP162.jpg  
  inflating: /content/dataset_fashion/Dress/SP163.jpg  
  inflating: /content/dataset_fashion/Dress/SP164.jpg  
  inflating: /content/dataset_fashion/Dress/SP165.jpg  
  inflating: /content/dataset_fashion/Dress/SP166.jpg  
  inflating: /content/dataset_fashion/Dress/SP167.jpg  
  inflating: /content/dataset_fashion/Dress/SP168.jpg  
  inflating: /content/dataset_fashion/Dress/SP169.jpg  
  inflating: /content/dataset_fashion/Dress/SP170.jpg  
  inflating: /content/dataset_fashion/Dress/SP171.jpg  
  inflating: /content/dataset_fashion/Dress/SP172.jpg  
  inflating: /content/dataset_fashion/Dres

**Tạo file fashion_resnet50.pth (Huấn luyện)**

In [2]:
# 1. Tiền xử lý ảnh
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 2. Load dữ liệu từ các thư mục (Dress, Hat, Pant...)
train_data = datasets.ImageFolder('/content/dataset_fashion', transform=transform)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

# 3. Khởi tạo mô hình
model = models.resnet50(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 6) # 6 lớp: Dress, Hat, Outerwear, Pant, Shirt, Shoes
device = torch.device("cuda")
model = model.to(device)

# 4. Huấn luyện nhanh (ví dụ 5 epochs)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(5):
    model.train()
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} hoàn tất.")

# --- TẠO FILE THỨ NHẤT ---
torch.save(model.state_dict(), 'fashion_resnet50.pth')
print("Đã tạo xong file fashion_resnet50.pth")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 88.5MB/s]


Epoch 1 hoàn tất.
Epoch 2 hoàn tất.
Epoch 3 hoàn tất.
Epoch 4 hoàn tất.
Epoch 5 hoàn tất.
Đã tạo xong file fashion_resnet50.pth


**Tạo file vector_db.index và product_ids.npy**

In [3]:
# 1. Chuyển mô hình sang chế độ trích xuất đặc trưng (bỏ lớp phân loại cuối)
feature_extractor = nn.Sequential(*list(model.children())[:-1])
feature_extractor.eval()

vectors = []
product_names = []

# 2. Quét toàn bộ ảnh trong dataset
# Lưu ý: lấy theo đúng thứ tự để id khớp với vector
for class_name in sorted(os.listdir('/content/dataset_fashion')):
    class_path = os.path.join('/content/dataset_fashion', class_name)
    if os.path.isdir(class_path):
        for img_file in os.listdir(class_path):
            img_path = os.path.join(class_path, img_file)
            try:
                img = Image.open(img_path).convert('RGB')
                img_t = transform(img).unsqueeze(0).to(device)

                with torch.no_grad():
                    vec = feature_extractor(img_t).flatten().cpu().numpy()

                vectors.append(vec)
                product_names.append(img_file) # Lưu tên file ảnh (ví dụ: SP401.jpg)
            except:
                continue

# --- TẠO FILE THỨ HAI: vector_db.index ---
vectors = np.array(vectors).astype('float32')
index = faiss.IndexFlatL2(2048) # ResNet50 trả về vector 2048 chiều
index.add(vectors)
faiss.write_index(index, "vector_db.index")

# --- TẠO FILE THỨ BA: product_ids.npy ---
np.save("product_ids.npy", np.array(product_names))

print("Đã tạo xong vector_db.index và product_ids.npy")

Đã tạo xong vector_db.index và product_ids.npy


**Tải 3 file về máy tính**

In [4]:
from google.colab import files

files.download('fashion_resnet50.pth')
files.download('vector_db.index')
files.download('product_ids.npy')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>